In [ ]:
# For downloading full step-by-step history from W&B

import wandb
import pandas as pd
from enum import StrEnum


class ProjectName(StrEnum):
    AEN_SAE_PYTHIA70M = "farischaudhry-imperial-college-london/aen-sae-llm-pythia70m"
    AEN_SAE_MECHANISTIC = "farischaudhry-imperial-college-london/aen-sae-mechanistic"
    AEN_SAE_LLAMA8B = "farischaudhry-imperial-college-london/aen-sae-llm-llama8b"

api = wandb.Api()

In [ ]:
def download_wandb_history(project_name: ProjectName, save_path: str | None = None) -> pd.DataFrame:
    runs = api.runs(project_name)
    print(f"Found {len(runs)} total runs. Starting full history extraction...")

    all_history_rows = []
    runs_processed = 0

    for run in runs:
        # Skip crashed or currently running sweeps
        if run.state != "finished":
            continue

        print(f"Downloading history for run: {run.name} ({run.id})...")

        history_iterator = run.scan_history()
        for step_data in history_iterator:
            # Create a new row starting with the run identifiers
            clean_row = {
                "run_id": run.id,
                "run_name": run.name,
            }

            # Filter out histograms and complex objects (keep only numbers and strings)
            for key, value in step_data.items():
                # If the value is a standard flat type, keep it. 
                # If it's a dict or list (like a W&B histogram), ignore it.
                if isinstance(value, (int, float, str, bool)):
                    clean_row[key] = value
                    
            all_history_rows.append(clean_row)

        runs_processed += 1

    print(f"Finished processing {runs_processed} runs. Total history rows collected: {len(all_history_rows)}")#
    history_df = pd.DataFrame(all_history_rows)

    if 'step' in history_df.columns:
        history_df = history_df.sort_values(by=["run_name", "step"])

    if save_path is None:
        save_path = f"{project_name.split('/')[-1]}_step_by_step_history.csv"
    history_df.to_csv(save_path, index=False)
    print(f"History saved to {save_path}")
    return history_df